<a href="https://colab.research.google.com/github/ingridmidory/Escuela-Nacional-Preparatoria-No.4-Vidal-Casta-eda-y-N-jera-/blob/main/Estad%C3%ADstica_TurnoGrado_de_horas_libres_por_d%C3%ADa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import zipfile
import glob
import os

In [ ]:
df = pd.read_excel("/content/Horarios 25-26.xlsx")

In [ ]:
# ==============================
# CONFIGURACIÓN
# ==============================

# Mapear días y horas
dias_map = {"LU":"Lunes","MA":"Martes","MI":"Miércoles","JU":"Jueves","VI":"Viernes"}
horas_map = {
    1:"07:00–07:50", 2:"07:50–08:40", 3:"08:40–09:30", 4:"09:30–10:20",
    5:"10:20–11:10", 6:"11:10–12:00", 7:"12:00–12:50", 8:"12:50–13:40",
    9:"13:40–14:30", 10:"14:30–15:20", 11:"15:20–16:10", 12:"16:10–17:00",
    13:"17:00–17:50", 14:"17:50–18:40", 15:"18:40–19:30", 16:"19:30–20:20",
    17:"20:20–21:10", 18:"21:10-22:00"
}
dias = list(dias_map.keys())
modulos = list(range(1,19))

dias = list(dias_map.keys())
modulos = list(range(1,19))

In [ ]:
# ==============================
# NORMALIZAR GRUPOS
# ==============================
def normalizar_grupo(grupo):
    return ''.join(filter(str.isdigit, str(grupo)))  # 401A -> 401

df['GRUPO_NORMAL'] = df['GRUPO'].apply(normalizar_grupo)

In [ ]:
# Rangos de turnos

turnos = {
    "matutino": list(range(1, 10)),   # módulos 1–9
    "vespertino": list(range(10, 18)) # módulos 10–17
}


In [ ]:
turno_por_grupo = {
    "401": "matutino",
    "402": "matutino",
    "403": "matutino",
    "404": "matutino",
    "405": "matutino",
    "406": "matutino",
    "407": "matutino",
    "408": "matutino",
    "409": "matutino",
    "410": "matutino",
    "411": "matutino",
    "412": "matutino",
    "413": "matutino",
    "414": "matutino",
    "415": "matutino",
    "416": "matutino",
    "417": "matutino",
    "501": "matutino",
    "502": "matutino",
    "503": "matutino",
    "504": "matutino",
    "505": "matutino",
    "506": "matutino",
    "507": "matutino",
    "508": "matutino",
    "509": "matutino",
    "510": "matutino",
    "511": "matutino",
    "512": "matutino",
    "513": "matutino",
    "514": "matutino",
    "515": "matutino",
    "516": "matutino",
    "517": "matutino",
    "601": "matutino",
    "602": "matutino",
    "603": "matutino",
    "604": "matutino",
    "605": "matutino",
    "606": "matutino",
    "607": "matutino",
    "608": "matutino",
    "609": "matutino",
    "610": "matutino",
    "611": "matutino",
    "612": "matutino",
    "613": "matutino",
    "614": "matutino",
    "615": "matutino",
    "451": "vespertino",
    "452": "vespertino",
    "453": "vespertino",
    "454": "vespertino",
    "455": "vespertino",
    "456": "vespertino",
    "457": "vespertino",
    "458": "vespertino",
    "459": "vespertino",
    "460": "vespertino",
    "461": "vespertino",
    "462": "vespertino",
    "463": "vespertino",
    "464": "vespertino",
    "551": "vespertino",
    "552": "vespertino",
    "553": "vespertino",
    "554": "vespertino",
    "555": "vespertino",
    "556": "vespertino",
    "557": "vespertino",
    "558": "vespertino",
    "559": "vespertino",
    "560": "vespertino",
    "561": "vespertino",
    "562": "vespertino",
    "563": "vespertino",
    "564": "vespertino",
    "651": "vespertino",
    "652": "vespertino",
    "653": "vespertino",
    "654": "vespertino",
    "655": "vespertino",
    "656": "vespertino",
    "657": "vespertino",
    "658": "vespertino",
    "659": "vespertino",
    "660": "vespertino",
    "661": "vespertino",
    "662": "vespertino",
    "663": "vespertino",
}

In [ ]:
# ==================================================
# FUNCIÓN PRINCIPAL PARA GENERAR LA ESTADÍSTICA
# ==================================================

def estadistica_horas_libres(df, turno_por_grupo, dias_map):
    # Normalizamos el grupo (por si tiene letras)
    df['GRUPO_NORMAL'] = df['GRUPO'].apply(lambda x: ''.join(filter(str.isdigit, str(x))))

    # Asignar turno usando tu diccionario
    df['TURNO'] = df['GRUPO_NORMAL'].map(turno_por_grupo)

    # Extraer día (ejemplo: LU01 → LU)
    df['DIA'] = df['HORA'].astype(str).str[:2]

    # Calcular grado (primer dígito del grupo)
    df['GRADO'] = df['GRUPO_NORMAL'].str[0]

    # Marcar horas libres
    df['HORA_LIBRE'] = df['RFC'].isna() | df['PROFESOR(A)'].str.contains("Prof.porasignar", case=False, na=False)

    # Filtramos solo los grupos válidos (que tengan turno definido)
    df = df[df['TURNO'].notna()]

    # ===============================
    # 1️⃣ Contar si un grupo tiene al menos una hora libre por día
    # ===============================
    resumen = (
        df.groupby(["TURNO", "GRADO", "DIA", "GRUPO_NORMAL"])["HORA_LIBRE"]
          .any()  # True si tiene al menos una hora libre ese día
          .reset_index()
    )

    # ===============================
    # 2️⃣ Contar grupos con horas libres por turno, grado y día
    # ===============================
    estadistica = (
        resumen.groupby(["TURNO", "GRADO", "DIA"])["HORA_LIBRE"]
        .agg(["sum", "count"])
        .reset_index()
        .rename(columns={"sum": "grupos_con_hora_libre", "count": "total_grupos"})
    )

    # Calcular porcentaje
    estadistica["porcentaje"] = (estadistica["grupos_con_hora_libre"] / estadistica["total_grupos"] * 100).round(2)

    # Traducir días al nombre completo
    estadistica["DIA"] = estadistica["DIA"].map(dias_map)

    # Ordenar resultado
    estadistica = estadistica.sort_values(["TURNO", "GRADO", "DIA"]).reset_index(drop=True)

    return estadistica

In [ ]:
# ==================================================
# USO
# ==================================================

# Suponiendo que ya tienes df cargado y los diccionarios definidos
resultado = estadistica_horas_libres(df, turno_por_grupo, dias_map)

# Mostrar resultado
display(resultado)

# Guardar en Excel si lo necesitas
resultado.to_excel("estadistica_horas_libres_por_turno_y_grado.xlsx", index=False)

,TURNO,GRADO,DIA,grupos_con_hora_libre,total_grupos,porcentaje
0,matutino,4,Jueves,2,17,11.76
1,matutino,4,Lunes,0,17,0.00
2,matutino,4,Martes,1,17,5.88
3,matutino,4,Miércoles,1,17,5.88
4,matutino,4,Viernes,1,17,5.88
5,matutino,5,Jueves,0,17,0.00
6,matutino,5,Lunes,0,17,0.00
7,matutino,5,Martes,0,17,0.00
8,matutino,5,Miércoles,1,17,5.88
9,matutino,5,Viernes,0,17,0.00


from matplotlib import pyplot as plt
resultado['grupos_con_hora_libre'].plot(kind='hist', bins=20, title='grupos_con_hora_libre')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
resultado['total_grupos'].plot(kind='hist', bins=20, title='total_grupos')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
resultado['porcentaje'].plot(kind='hist', bins=20, title='porcentaje')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
resultado.groupby('TURNO').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(resultado['GRADO'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(resultado, x='grupos_con_hora_libre', y='GRADO', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(resultado['DIA'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(resultado, x='grupos_con_hora_libre', y='DIA', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(resultado['TURNO'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(resultado, x='total_grupos', y='TURNO', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)